# Level 2, Notebook 04: Support and Operations Signals

**Problems in this notebook**

- Problem 14: Parse Logs and Detect Slow Tenants
- Problem 16: Customer Escalation Detection

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill in the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `04_support_and_ops_signals_THEORY.md`, next to this notebook.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 14: Parse Logs and Detect Slow Tenants

**The situation**

A customer says the assistant feels slow. You have request logs, each with a tenant ID and a latency in milliseconds. Some lines are broken or missing fields. You need a quick list of which tenants are really slow.

**Your task**

Write `detect_slow_tenants(logs, threshold_ms, min_samples=1)`. It returns the sorted tenant IDs whose average latency is above the threshold, counting only tenants with at least `min_samples` valid lines.

**Example**

`[{"tenant": "a", "latency_ms": 1000}, {"tenant": "a", "latency_ms": 500}]`, threshold `700` -> `["a"]`

**Ask before coding**

- Should we use the average, or a percentile like p95?
- What time window do these logs cover?
- How many requests do we need before we call a tenant slow?

**Plan to say aloud**

1. Keep a running total and a count per tenant.
2. Skip lines with no tenant or a latency that is not a real number.
3. After the loop, work out each tenant's average.
4. Keep tenants with enough samples and an average above the threshold.
5. Return them sorted.

### Prompt for your AI assistant

```text
Write a Python function `detect_slow_tenants(logs: list[dict], threshold_ms: float, min_samples: int = 1) -> list[str]`. Each log may have "tenant" (str) and "latency_ms" (number). Skip a log if tenant is missing or empty, or if latency_ms is not an int or float (treat bool as invalid). For each tenant, average the latency over its valid logs. Return tenants with count >= min_samples and average strictly greater than threshold_ms, sorted ascending. Never raise on a malformed log line. Show only the function.
```

**Check the AI's code for**

- Uses `log["latency_ms"]` and crashes on the first broken line.
- Accepts strings like `"900"` or booleans as latencies. In Python, `True` is an int.
- Uses `>=` instead of `>`, flagging tenants that sit exactly on the threshold.
- Divides by all lines for a tenant, including skipped ones, which drags the average down.

In [ ]:
def detect_slow_tenants(logs: list[dict], threshold_ms: float, min_samples: int = 1) -> list[str]:
    """Return sorted tenants whose average latency is above threshold_ms.

    Rules:
    - Skip a log with a missing/empty "tenant" or a "latency_ms" that is not int/float (bool is invalid).
    - Average over valid logs only.
    - Flag a tenant only if it has >= min_samples valid logs and average > threshold_ms.
    - Never raise on a bad line.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"tenant": "a", "latency_ms": 1000}, {"tenant": "a", "latency_ms": 500}], 700), ["a"]),
    (([{"tenant": "a", "latency_ms": 500}], 500), []),
    (([{"tenant": "a"}, {"latency_ms": 900}, {"tenant": "b", "latency_ms": "900"},
       {"tenant": "b", "latency_ms": True}, {"tenant": "b", "latency_ms": 900}], 500), ["b"]),
    (([{"tenant": "a", "latency_ms": 2000}], 500, 2), []),
    (([{"tenant": "c", "latency_ms": 800}, {"tenant": "a", "latency_ms": 900}, {"tenant": "b", "latency_ms": 100}],
      500), ["a", "c"]),
    (([], 500), []),
]
run_checks(detect_slow_tenants, cases)

### Reference solution

Try first. Then compare.

In [ ]:
from collections import defaultdict


def detect_slow_tenants_reference(logs: list[dict], threshold_ms: float, min_samples: int = 1) -> list[str]:
    totals: dict[str, float] = defaultdict(float)
    counts: dict[str, int] = defaultdict(int)
    for log in logs:
        tenant = log.get("tenant")
        latency = log.get("latency_ms")
        if not tenant or isinstance(latency, bool) or not isinstance(latency, (int, float)):
            continue  # skip broken lines instead of crashing
        totals[tenant] += latency
        counts[tenant] += 1
    return sorted(
        tenant for tenant in totals
        if counts[tenant] >= min_samples and totals[tenant] / counts[tenant] > threshold_ms
    )


assert run_checks(detect_slow_tenants_reference, cases)

**Explain it in 30 seconds**

> "I make one pass over the logs, keeping a total and a count per tenant, and I skip any line missing a tenant or a numeric latency instead of crashing. Then I work out averages and flag tenants above the threshold that have enough samples to trust. I return a sorted list so the report is stable. For a real investigation I would look at p95 and traces, because an average can hide a slow tail."

**Edge cases to mention**

- A tenant exactly on the threshold is not flagged (strictly greater).
- One slow request can flag a small tenant. min_samples guards against that.
- Booleans and numeric strings are skipped as invalid latencies.
- Failed requests can be very fast and pull the average down, hiding a real problem.

**If they ask "how would this work in production?"**

- Use p95 and p99 latency as well as averages, so a slow tail is visible.
- Split latency into model, retrieval and connector time using request traces.
- Put per-tenant latency on a dashboard with alerts tied to the customer's SLA.

## Problem 16: Customer Escalation Detection

**The situation**

A support assistant answers routine questions on its own. Messages that are urgent, threaten to cancel, or mention legal action must reach a human fast. Missing one can lose a customer, and flagging too many buries the support team.

**Your task**

Write `should_escalate(message, triggers=ESCALATION_TRIGGERS)`. It returns `(True, trigger)` for the first trigger found in the message, or `(False, "")` if none match. Returning the reason helps the support team trust the decision.

**Example**

`"This is urgent, we will cancel our contract"` -> `(True, "urgent")`

**Ask before coding**

- Which words or phrases must trigger an escalation?
- Do some customers, like VIP accounts, escalate on weaker signals?
- Should we return why we escalated?

**Plan to say aloud**

1. Lowercase the message and squash extra spaces.
2. For each trigger, look for it at the start of a word.
3. Return True and the first trigger that matches.
4. If nothing matches, return False and an empty reason.

### Prompt for your AI assistant

```text
Write a Python function `should_escalate(message: str, triggers: tuple[str, ...] = ESCALATION_TRIGGERS) -> tuple[bool, str]`. Lowercase the message and collapse runs of whitespace to one space. Check the triggers in order. A trigger matches if it appears starting at a word boundary (use `re.search` with `\b` plus `re.escape(trigger)`), so "cancel" matches "cancellation" but "urgent" does not match "detergent". Return (True, trigger) for the first match, else (False, ""). Show only the function.
```

**Check the AI's code for**

- Uses plain `in`, so `urgent` matches inside `detergent` and raises false alarms.
- Uses a long phrase like `cancel contract`, which misses `cancel our contract`. Short triggers are safer.
- Forgets `re.escape`, so a trigger with a special character breaks the pattern.
- Returns only a bool, losing the reason the support team needs.

In [ ]:
import re

ESCALATION_TRIGGERS = ("urgent", "cancel", "lawsuit", "legal action", "lawyer", "escalate")


def should_escalate(message: str, triggers: tuple[str, ...] = ESCALATION_TRIGGERS) -> tuple[bool, str]:
    """Return (True, trigger) for the first trigger found, else (False, "").

    Rules:
    - Lowercase the message and collapse whitespace to single spaces.
    - Check triggers in order. A trigger must start at a word boundary:
      "cancel" matches "cancellation", "urgent" does not match "detergent".
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("This is URGENT, we will cancel our contract",), (True, "urgent")),
    (("How do I reset my password?",), (False, "")),
    (("We are considering legal\n   action",), (True, "legal action")),
    (("Please process my cancellation",), (True, "cancel")),
    (("The detergent order arrived",), (False, "")),
    (("I want a refund", ("refund",)), (True, "refund")),
    (("",), (False, "")),
]
run_checks(should_escalate, cases)

### Reference solution

Try first. Then compare.

In [ ]:
import re

ESCALATION_TRIGGERS = ("urgent", "cancel", "lawsuit", "legal action", "lawyer", "escalate")


def should_escalate_reference(message: str, triggers: tuple[str, ...] = ESCALATION_TRIGGERS) -> tuple[bool, str]:
    text = " ".join(message.lower().split())
    for trigger in triggers:
        if re.search(r"\b" + re.escape(trigger), text):
            return True, trigger
    return False, ""


assert run_checks(should_escalate_reference, cases)

**Explain it in 30 seconds**

> "I clean the message first, lowercase with single spaces, then check a short list of trigger words in order. A trigger has to start at a word boundary, so "urgent" does not fire on "detergent", but "cancel" still catches "cancellation". I return the decision plus the trigger that fired, so a human can see why. It is a baseline; in production I would combine rules with a classifier and customer context."

**Edge cases to mention**

- Sarcasm like "great, just great" is missed by keywords.
- Messages in other languages miss English triggers.
- Quoted text, like a pasted contract clause about legal action, can cause a false alarm.
- An empty message returns (False, "").

**If they ask "how would this work in production?"**

- Combine rules with a classifier, plus customer tier, open SLA breaches and message history.
- Let each customer set their own triggers. The function already takes them as a parameter.
- Measure missed escalations and false alarms from human feedback, and tune the triggers from those numbers.

## Wrap-up

- Both turn messy, real-world input (log lines, customer messages) into a clear yes or no signal.
- Both trade false alarms against misses. Say which one hurts the customer more.
- Both return something a human can act on and trust: a sorted list, or a decision with its reason.